# AMR Project - Step 4: XGBoost Classifier on Advanced Features (NCBI Dataset)

This notebook trains an XGBoost classifier with class balancing on the preprocessed 140 features.

### Objectives:
1. **Split Dataset**: Perform the same stratified 80/20 train/test split on `preprocessed_data.csv`.
2. **Column Sanitization**: Sanitize all column names to be alphanumeric and unique, preventing DMatrix compile errors in XGBoost.
3. **Address Class Imbalance**: Incorporate `scale_pos_weight` to weight minority class classification errors.
4. **Train XGBoost**: Train an XGBoost model using the fast histogram-based tree method (`tree_method='hist'`).
5. **Evaluation**: Compute Accuracy, Precision, Recall, F1 score, and compare against Logistic Regression baselines.

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import xgboost as xgb

## 1. Load Preprocessed Data

In [2]:
preprocessed_path = "../data/processed/preprocessed_data.csv"
print(f"Loading data from {preprocessed_path}...")
df = pd.read_csv(preprocessed_path, low_memory=False)
print(f"Dataset shape: {df.shape}")

Loading data from ../data/processed/preprocessed_data.csv...
Dataset shape: (401831, 141)


## 2. Sanitize Column Names

In [3]:
def sanitize_column_names(columns):
    seen = {}
    sanitized = []
    for col in columns:
        clean_col = re.sub(r'[^a-zA-Z0-9]', '_', str(col))
        clean_col = re.sub(r'_+', '_', clean_col).strip('_')
        if clean_col in seen:
            seen[clean_col] += 1
            clean_col = f"{clean_col}_{seen[clean_col]}"
        else:
            seen[clean_col] = 0
        sanitized.append(clean_col)
    return sanitized

X = df.drop(columns=['Target'])
y = df['Target']

sanitized_features = sanitize_column_names(X.columns)
X.columns = sanitized_features
print("Sanitized first 10 columns:", X.columns[:10].tolist())

Sanitized first 10 columns: ['MIC_mg_L', 'Organism_group_Acinetobacter_baumannii', 'Organism_group_Campylobacter_jejuni', 'Organism_group_E_coli_and_Shigella', 'Organism_group_Klebsiella_pneumoniae', 'Organism_group_Mycobacterium_tuberculosis', 'Organism_group_Neisseria_gonorrhoeae', 'Organism_group_Pseudomonas_aeruginosa', 'Organism_group_Salmonella_enterica', 'Organism_group_Staphylococcus_aureus']


## 3. Train/Test Stratified Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")

Train size: 321464 | Test size: 80367


## 4. Train XGBoost Classifier

In [5]:
scale_pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)
print(f"Calculated scale_pos_weight: {scale_pos_weight:.4f}")

model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost on preprocessed features...")
model.fit(X_train, y_train)
print("Model trained.")

Calculated scale_pos_weight: 3.1781
Training XGBoost on preprocessed features...
Model trained.


## 5. Evaluation

In [6]:
y_pred = model.predict(X_test)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:   ", recall_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Susceptible', 'Resistant']))

Accuracy:  0.9839113068796894
Precision: 0.9488642049434605
Recall:    0.9859110995580972
F1 Score:  0.967032967032967

Confusion Matrix:
[[60110  1022]
 [  271 18964]]

Classification Report:
              precision    recall  f1-score   support

 Susceptible       1.00      0.98      0.99     61132
   Resistant       0.95      0.99      0.97     19235

    accuracy                           0.98     80367
   macro avg       0.97      0.98      0.98     80367
weighted avg       0.98      0.98      0.98     80367



## 6. Save Model

In [7]:
xgb_model_path = "../models/xgb_model_new.pkl"
with open(xgb_model_path, 'wb') as f:
    pickle.dump(model, f)
print("XGBoost model saved successfully.")

XGBoost model saved successfully.
